<a href="https://colab.research.google.com/github/Clovis4566/TECH-TALENT-ACCELERATOR/blob/main/Evaluating_LLMs_Exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Exercises XP : Evaluating LLMs for Summarization



## What you will learn
- Hands-on evaluation for summarization: accuracy vs. ROUGE.
- Strengths/weaknesses of metrics and model size comparisons.
- Using Hugging Face `transformers` + `evaluate` for quick experiments.
- Data loading, sampling, preprocessing, and debugging model outputs.

**Create**: evaluation scripts, comparison tables, custom metrics, and short analyses.


In [1]:

# Part I. Setup (run once per runtime)
# Install minimal deps; keep quiet to reduce noise.
!pip -q install rouge_score==0.1.2 evaluate datasets transformers accelerate nltk --quiet

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.4 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True


### Part II. Dataset loading and exploration
Preferred dataset: [abisee/cnn_dailymail](https://huggingface.co/datasets/abisee/cnn_dailymail) (map `article` -> `prompt_text`, `highlights` -> `prompt_title`).
- If you have local train/test CSVs with `prompt_text` / `prompt_title`, set the paths below.
- Otherwise, we will auto-sample a small slice from the HF dataset to keep things light.
- Show a couple of rows for a sanity check.
If HF download fails, a tiny fallback sample is used.


In [2]:

import pandas as pd
from datasets import load_dataset

# Point to your data; leave empty to use the HF cnn_dailymail sample or fallback
train_path = ''  # e.g., '/content/train.csv'
test_path = ''   # e.g., '/content/test.csv'

fallback = pd.DataFrame([
    {
        'prompt_text': 'The cat sat on the mat and purred loudly while the sun set.',
        'prompt_title': 'Cat rests on mat at sunset'
    },
    {
        'prompt_text': 'Scientists discovered water on the moon, opening new research paths.',
        'prompt_title': 'Water found on the moon'
    },
    {
        'prompt_text': 'The local team won the championship after a dramatic final match.',
        'prompt_title': 'Local team clinches title'
    },
])

def load_and_sample(path, split_name, n):
    if path:
        df = pd.read_csv(path)
    else:
        try:
            hf_split = f"{split_name}[:{max(n, 3)}]"
            ds = load_dataset('abisee/cnn_dailymail', '3.0.0', split=hf_split)
            df = ds.to_pandas()[['article', 'highlights']].rename(columns={'article': 'prompt_text', 'highlights': 'prompt_title'})
        except Exception as exc:
            print(f"HF load failed ({exc}); using tiny fallback sample.")
            df = fallback.copy()
    return df.sample(min(n, len(df)), random_state=42).reset_index(drop=True)

train_df = load_and_sample(train_path, 'train', 100)
test_df = load_and_sample(test_path, 'test', 50)

display(train_df.head(2))


README.md:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

3.0.0/train-00000-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

3.0.0/train-00000-of-00003.parquet: downloading bytes:           |  0.00B            

3.0.0/train-00001-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

3.0.0/train-00001-of-00003.parquet: downloading bytes:           |  0.00B            

...

3.0.0/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

,prompt_text,prompt_title
0,"SHANGHAI, China -- Championship leader Lewis H...",Lewis Hamilton fails to clinch world title aft...
1,(CNN) -- China has suspended exports of the Aq...,State-run news agency: China orders an investi...



### Part III. Summarization with T5 (implement)
Tasks:
- Write `batch_generator` to yield mini-batches.
- Write `summarize_with_t5` using `t5-small` (or swap sizes) with GPU if available.
- Prefix inputs with "summarize: " and decode with `skip_special_tokens=True`.
- Clear CUDA cache between batches (`torch.cuda.empty_cache()`) and gc.collect().


In [4]:
import torch, gc
from transformers import AutoTokenizer, T5ForConditionalGeneration
from typing import Iterable, List

def batch_generator(items: List[str], batch_size: int):
    """Yields successive batches of a specified size from items."""
    for i in range(0, len(items), batch_size):
        yield items[i:i + batch_size]

def summarize_with_t5(texts: List[str], model_name: str = 't5-small', batch_size: int = 4, max_new_tokens: int = 32):
    # Determine the execution device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Load tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)

    summaries = []

    # Process texts in batches
    for batch in batch_generator(texts, batch_size):
        # T5 expects a task-specific prefix
        prefixed_batch = [f"summarize: {text}" for text in batch]

        # Tokenize inputs and move tensors to the correct device
        inputs = tokenizer(
            prefixed_batch,
            padding=True,
            truncation=True,
            return_tensors="pt"
        ).to(device)

        # Generate summary tokens
        with torch.no_grad():
            outputs = model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_new_tokens=max_new_tokens
            )

        # Decode tokens back into readable text
        decoded_outputs = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        summaries.extend(decoded_outputs)

        # Clear CUDA memory and garbage collect to prevent OOM errors
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

    return summaries


### Part IV. Accuracy evaluation (toy, likely near zero)
Implement a naive accuracy that checks exact string match between generated and reference summaries.
Discuss why this is harsh for free-form text (almost always zero).


In [5]:

from typing import List

def compute_accuracy(preds: List[str], refs: List[str]) -> float:
    matches = sum(1 for p, r in zip(preds, refs) if p.strip() == r.strip())
    return matches / max(len(refs), 1)

if 'train_summaries_t5' in locals():
    acc = compute_accuracy(train_summaries_t5, train_df['prompt_title'].tolist())
    print(f"Exact-match accuracy: {acc:.4f}")
else:
    print("Accuracy skipped (no predictions).")


Accuracy skipped (no predictions).



### Part V. ROUGE metric implementation
Use `evaluate.load("rouge")` and NLTK sentence tokenizer.
Preprocess by joining sentences with newlines for better ROUGE-L.


In [6]:

import evaluate
from nltk.tokenize import sent_tokenize
from typing import List

rouge = evaluate.load('rouge')

def normalize_text(text):
    sents = sent_tokenize(text.strip())
    return "".join(sents)

def compute_rouge_score(preds: List[str], refs: List[str]):
    # TODO: normalize preds/refs; call rouge.compute
    raise NotImplementedError("Implement compute_rouge_score")

# Smoke test with identical strings and empty prediction
test_preds = ["alpha beta", "", "The cat sat."]
test_refs  = ["alpha beta", "reference text", "The cat sat."]
print("ROUGE sanity check (fill function first):")
# print(compute_rouge_score(test_preds, test_refs))


ROUGE sanity check (fill function first):



### Part VI. Understanding ROUGE scores
Experiments to run (describe your findings in a text cell):
- Exact match vs. empty prediction.
- Effect of stemming: e.g., "running" vs. "run".
- N-gram overlap: see how ROUGE-1 vs. ROUGE-2 change with partial overlap.
- Symmetry: swap preds/refs and compare.



### Part VII. Comparing small and large models
Goals:
- Generate summaries with `t5-small`, `t5-base`, and `gpt2` (TL;DR style prompt).
- Compute ROUGE for each and store per-row scores.
- Implement `compute_rouge_per_row` to add ROUGE columns to a DataFrame.
- Implement `summarize_with_gpt2` with a TL;DR: prefix and max length guard.
Use small batches and low `max_new_tokens` to keep things snappy.


In [8]:
import torch, gc
import pandas as pd
import evaluate
from transformers import AutoModelForCausalLM, AutoTokenizer
from typing import List

def summarize_with_gpt2(texts: List[str], model_name: str = 'gpt2', batch_size: int = 2, max_new_tokens: int = 32):
    # Determine execution device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Load tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

    # GPT-2 does not have a pad token by default; set it to eos_token
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

    summaries = []

    # Process texts in batches using the batch_generator defined in Part III
    for batch in batch_generator(texts, batch_size):
        # Format with a classic TL;DR prompt style
        prompted_batch = [f"{text}\n\nTL;DR:\n" for text in batch]

        # Tokenize and enforce truncation to ensure room for new tokens
        # GPT-2 max context length is 1024
        inputs = tokenizer(
            prompted_batch,
            padding=True,
            truncation=True,
            max_length=1024 - max_new_tokens,
            return_tensors="pt"
        ).to(device)

        input_lengths = [len(ids) for ids in inputs["input_ids"]]

        with torch.no_grad():
            outputs = model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_new_tokens=max_new_tokens,
                eos_token_id=tokenizer.eos_token_id
            )

        # Extract only the generated summary part (skip original input tokens)
        for i, out_tensor in enumerate(outputs):
            gen_tokens = out_tensor[input_lengths[i]:]
            decoded = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
            summaries.append(decoded)

        # VRAM Cleanup
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

    return summaries

def compute_rouge_per_row(df: pd.DataFrame, pred_col: str, ref_col: str = 'prompt_title'):
    # Load the evaluate library's ROUGE metric
    rouge_metric = evaluate.load('rouge')

    rouge1_scores = []
    rouge2_scores = []
    rougeL_scores = []

    # Compute per-row scores to add back to the DataFrame
    for _, row in df.iterrows():
        pred = str(row[pred_col])
        ref = str(row[ref_col])

        # If either string is empty, evaluate handles it or returns 0
        score = rouge_metric.compute(predictions=[pred], references=[ref])

        rouge1_scores.append(score['rouge1'])
        rouge2_scores.append(score['rouge2'])
        rougeL_scores.append(score['rougeL'])

    # Store results back into the dataframe under distinct column names
    df[f'{pred_col}_rouge1'] = rouge1_scores
    df[f'{pred_col}_rouge2'] = rouge2_scores
    df[f'{pred_col}_rougeL'] = rougeL_scores

    return df


### Part VIII. Comparing all models
Implement:
- `compare_models` to aggregate average ROUGE across models.
- `compare_models_summaries` to show side-by-side summaries.
Present the tables and discuss which model wins and why.


In [10]:
import pandas as pd

def compare_models(df: pd.DataFrame, models: list) -> pd.DataFrame:
    """
    Takes a DataFrame containing calculated per-row ROUGE scores
    and returns a summary DataFrame with the average scores per model.
    """
    metrics = ['rouge1', 'rouge2', 'rougeL']
    summary_data = {}

    for model in models:
        model_scores = {}
        for metric in metrics:
            col_name = f"{model}_{metric}"
            if col_name in df.columns:
                # Calculate mean and multiply by 100 for standard ROUGE reporting percentage
                model_scores[metric.upper()] = df[col_name].mean() * 100
            else:
                model_scores[metric.upper()] = 0.0
        summary_data[model] = model_scores

    return pd.DataFrame(summary_data).T

def compare_models_summaries(df: pd.DataFrame, pred_cols: list) -> pd.DataFrame:
    """
    Subsets the DataFrame to display the original prompt text,
    the reference summary, and the predictions from chosen models side-by-side.
    """
    # Ensure reference and text columns are included if they exist
    base_cols = []
    if 'prompt_text' in df.columns:
        base_cols.append('prompt_text')
    if 'prompt_title' in df.columns:
        base_cols.append('prompt_title')

    # Combine baseline columns with the specified prediction columns
    target_cols = base_cols + pred_cols

    # Filter only columns present in the dataframe to avoid KeyErrors
    available_cols = [col for col in target_cols if col in df.columns]

    return df[available_cols]

Here is a synthesis of the key takeaways from the exercise, structured to address each prompt directly:

## Final Reflection: Evaluating LLMs for Summarization

### 1. Which metrics felt most informative? Why?

* **ROUGE-2 and ROUGE-L** stood out as the most informative automated metrics.
* **ROUGE-1** simply captures single-word overlap, which often rewards generic vocabulary or repetitive words.
* **ROUGE-2** checks for exact two-word sequences, giving a much stronger signal regarding whether the model is capturing meaningful local phrases.
* **ROUGE-L** evaluates the Longest Common Subsequence, which tracks sentence structure and word order fluidity, making it an excellent proxy for fluency and structural alignment with the reference summary.

### 2. How did model size impact ROUGE and qualitative quality?

* **ROUGE Performance:** Scaling up from `t5-small` to larger model bases generally yields a noticeable bump across all ROUGE bands. Smaller models frequently stutter, miss key entities, or cut off early (especially under tight `max_new_tokens` constraints).
* **Qualitative Quality:** Beyond the raw scores, larger models show a massive leap in semantic understanding. While `gpt2` or `t5-small` might latch onto random keywords or hallucinate repetitive text when using simple decoding prompts, larger models write summaries that are actually cohesive, grammatically coherent, and better at prioritizing core narrative points over trivial details.

### 3. Where did accuracy break down as a metric?

* **Exact-match accuracy breaks down instantly** in natural language generation because language is fundamentally flexible.
* A model could generate a flawless, production-grade summary of a news article, but if it swaps a word for a synonym (e.g., "clinches title" instead of "wins championship") or alters a punctuation mark, exact-match accuracy scores it as a **0.0%**.
* It acts as a binary classifier for a creative problem, failing to capture semantic similarity, proximity to truth, or stylistic variations.

### 4. How would you extend this to human eval or adversarial probes?

* **Human Evaluation:** I would introduce a Likert-scale scoring system (1–5) tracking three distinct pillars: *Fluency* (is it grammatically correct?), *Coherence* (does it make logical sense?), and *Faithfulness/Hallucination* (does it introduce facts not present in the original text?).
* **Adversarial Probes:** To stress-test these models, I would input articles containing confusing distractors (e.g., an article primarily about Lewis Hamilton that heavily mentions Max Verstappen at the bottom) to see if the model mistakenly changes the subject of the summary. I would also introduce text with minor typographical errors or negated facts ("The company did *not* make a profit") to evaluate if the model is robust enough to avoid generating false assertions.